In [2]:
!pip -q install uv

!uv python install 3.11
!uv venv --python 3.11 /content/tfenv

!uv pip install \
  --python /content/tfenv/bin/python \
  --index-url https://download.pytorch.org/whl/cpu \
  "torch==2.2.2"

!uv pip install \
  --python /content/tfenv/bin/python \
  "numpy==1.26.4" \
  "scipy==1.12.0" \
  "scikit-learn==1.4.2" \
  "dgl==1.1.3"

!/content/tfenv/bin/python -c "import torch, dgl, numpy; print('Python/DGL setup successful'); print('Torch:', torch.__version__); print('DGL:', dgl.__version__); print('NumPy:', numpy.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 42.4 MB/s eta 0:00:00
Installed Python 3.11.16 in 1.58s
 + cpython-3.11.16-linux-x86_64-gnu (python3.11)
Using CPython 3.11.16
Creating virtual environment at: tfenv
Activate with: source tfenv/bin/activate
Using Python 3.11.16 environment at: tfenv
Resolved 9 packages in 764ms
Prepared 9 packages in 5.02s
Installed 9 packages in 264ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + sympy==1.14.0
 + torch==2.2.2+cpu
 + typing-extensions==4.16.0
Using Python 3.11.16 environment at: tfenv
Resolved 15 packages in 410ms
Prepared 14 packages in 1.46s
Installed 14 packages in 100ms
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + cloudpickle==3.1.2
 + dgl==1.1.3
 + idna==3.19
 + joblib==1.6.0
 + numpy==1.26.4
 + psutil==7.2.2
 + requests==2.34.2
 + scikit-learn==1.4.2
 + scipy==1.12.0
 + threadpoolctl==3.6.0
 + tqdm==4.70.0
 + urllib3==2.7.0
DGL backend not selec

In [3]:
!uv pip install \
  --python /content/tfenv/bin/python \
  packaging setuptools wheel

Using Python 3.11.16 environment at: tfenv
Resolved 3 packages in 120ms
Prepared 3 packages in 45ms
Installed 3 packages in 9ms
 + packaging==26.3
 + setuptools==84.0.0
 + wheel==0.48.0


In [4]:
!DGLBACKEND=pytorch /content/tfenv/bin/python -c \
"import dgl, torch, numpy; print('DGL:', dgl.__version__); print('Torch:', torch.__version__); print('NumPy:', numpy.__version__)"

DGL: 1.1.3
Torch: 2.2.2+cpu
NumPy: 1.26.4


In [5]:
%%bash

export DGLBACKEND=pytorch

/content/tfenv/bin/python - <<'PY'
import dgl
import torch
import numpy as np
from sklearn.model_selection import train_test_split

path = "/content/dataset/tfinance"

graphs, metadata = dgl.load_graphs(path)
g = graphs[0]

features = g.ndata["feature"]
raw_labels = g.ndata["label"]

if raw_labels.ndim == 2:
    labels = raw_labels.argmax(dim=1).long()
else:
    labels = raw_labels.long().squeeze()

n = g.num_nodes()
e = g.num_edges()
feature_count = features.shape[1]

normal = int((labels == 0).sum())
fraud = int((labels == 1).sum())
unlabelled = int(((labels != 0) & (labels != 1)).sum())

print("=" * 70)
print("T-FINANCE BASIC STATISTICS")
print("=" * 70)
print("Number of nodes:", n)
print("Number of stored edge entries:", e)
print("Number of features:", feature_count)
print("Normal nodes:", normal)
print("Fraud/anomaly nodes:", fraud)
print("Unlabelled nodes:", unlabelled)
print(f"Fraud percentage: {100 * fraud / (normal + fraud):.6f}%")
print(f"Imbalance ratio normal:fraud = {normal / fraud:.6f}:1")
print("Node types:", g.ntypes)
print("Number of node types:", len(g.ntypes))
print("Canonical edge types:", g.canonical_etypes)
print("Number of relation/edge types:", len(g.canonical_etypes))
print("Feature tensor shape:", tuple(features.shape))
print("Stored label tensor shape:", tuple(raw_labels.shape))
print("Node fields:", list(g.ndata.keys()))
print("Edge fields:", list(g.edata.keys()))

src, dst = g.edges(order="eid")

valid = (
    ((labels[src] == 0) | (labels[src] == 1))
    & ((labels[dst] == 0) | (labels[dst] == 1))
)

self_loop = src == dst
different = labels[src] != labels[dst]

different_all = int((valid & different).sum())
same_all = int((valid & ~different).sum())
total_all = int(valid.sum())

nonself = valid & ~self_loop
different_nonself = int((nonself & different).sum())
same_nonself = int((nonself & ~different).sum())
total_nonself = int(nonself.sum())

print("\n" + "=" * 70)
print("GLOBAL HETEROPHILY")
print("=" * 70)
print("Total labelled edge entries:", total_all)
print("Different-label edge entries:", different_all)
print("Same-label edge entries:", same_all)
print("Self-loop entries:", int(self_loop.sum()))
print(f"Global heterophily including self-loops: {different_all / total_all:.8f}")
print(f"Non-self labelled edges: {total_nonself}")
print(f"Different-label non-self edges: {different_nonself}")
print(f"Same-label non-self edges: {same_nonself}")
print(f"Global heterophily excluding self-loops: {different_nonself / total_nonself:.8f}")
print(f"Global heterophily percentage: {100 * different_nonself / total_nonself:.4f}%")

degree = torch.bincount(src[nonself], minlength=n)
heterophilic_degree = torch.bincount(
    src[nonself & different],
    minlength=n
)

has_neighbours = degree > 0

local = (
    heterophilic_degree[has_neighbours].float()
    / degree[has_neighbours].float()
)

print("\n" + "=" * 70)
print("LOCAL HETEROPHILY")
print("=" * 70)
print("Nodes with labelled non-self neighbours:", int(has_neighbours.sum()))
print("Nodes without outgoing non-self neighbours:", int((~has_neighbours).sum()))
print(f"Mean local heterophily: {local.mean().item():.8f}")
print(f"Median local heterophily: {local.median().item():.8f}")
print(f"Local heterophily standard deviation: {local.std(unbiased=False).item():.8f}")

local_all = torch.full((n,), float("nan"))
local_all[has_neighbours] = local

for value, name in [(0, "NORMAL"), (1, "FRAUD/ANOMALY")]:
    mask = (labels == value) & has_neighbours
    values = local_all[mask]

    print(f"\n{name} NODES")
    print("Count:", len(values))
    print(f"Mean: {values.mean().item():.8f}")
    print(f"Median: {values.median().item():.8f}")
    print(f"Standard deviation: {values.std(unbiased=False).item():.8f}")

print("\n" + "=" * 70)
print("STORED SPLITS")
print("=" * 70)

available_masks = [
    name for name in ["train_mask", "val_mask", "test_mask"]
    if name in g.ndata
]

if available_masks:
    for name in available_masks:
        print(name, int(g.ndata[name].sum()))
else:
    print("No official split masks are stored in the T-Finance file.")
    print("The BWGNN code generates random stratified splits at runtime.")

indices = np.arange(n)
y = labels.numpy()

for train_ratio in [0.40, 0.01]:
    train_idx, remaining_idx, y_train, y_remaining = train_test_split(
        indices,
        y,
        train_size=train_ratio,
        stratify=y,
        random_state=2,
        shuffle=True
    )

    validation_idx, test_idx = train_test_split(
        remaining_idx,
        test_size=0.67,
        stratify=y_remaining,
        random_state=2,
        shuffle=True
    )

    print(f"\nRandom stratified split with {train_ratio:.0%} training:")
    print("Train:", len(train_idx))
    print("Validation:", len(validation_idx))
    print("Test:", len(test_idx))
    print(
        "Percentages:",
        f"{100 * len(train_idx) / n:.4f}% train,",
        f"{100 * len(validation_idx) / n:.4f}% validation,",
        f"{100 * len(test_idx) / n:.4f}% test"
    )

print("\nT-Finance analysis completed successfully.")
PY

T-FINANCE BASIC STATISTICS
Number of nodes: 39357
Number of stored edge entries: 42445086
Number of features: 10
Normal nodes: 37554
Fraud/anomaly nodes: 1803
Unlabelled nodes: 0
Fraud percentage: 4.581142%
Imbalance ratio normal:fraud = 20.828619:1
Node types: ['_N']
Number of node types: 1
Canonical edge types: [('_N', '_E', '_N')]
Number of relation/edge types: 1
Feature tensor shape: (39357, 10)
Stored label tensor shape: (39357, 2)
Node fields: ['feature', 'label']
Edge fields: []

GLOBAL HETEROPHILY
Total labelled edge entries: 42445086
Different-label edge entries: 1240480
Same-label edge entries: 41204606
Self-loop entries: 0
Global heterophily including self-loops: 0.02922553
Non-self labelled edges: 42445086
Different-label non-self edges: 1240480
Same-label non-self edges: 41204606
Global heterophily excluding self-loops: 0.02922553
Global heterophily percentage: 2.9226%

LOCAL HETEROPHILY
Nodes with labelled non-self neighbours: 39357
Nodes without outgoing non-self neighbo